In [27]:
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_core.messages import HumanMessage

In [28]:
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

In [29]:
class JokeState(TypedDict):
  topic:str
  joke:str
  explanation:str

In [30]:
def generate_joke(state: JokeState):

    topic = state["topic"]

    response = llm.invoke(
        [
            HumanMessage(
                content=f"Generate a short funny joke about {topic}. Only return the joke."
            )
        ]
    )

    return {
        "joke": response.content
    }


def explain_joke(state: JokeState):

    joke = state["joke"]

    response = llm.invoke(
        [
            HumanMessage(
                content=f"Explain this joke in simple words:\n{joke}"
            )
        ]
    )

    return {
        "explanation": response.content
    }

In [31]:
builder = StateGraph(JokeState)

builder.add_node("generate_joke", generate_joke)
builder.add_node("explain_joke", explain_joke)

builder.add_edge(START, "generate_joke")
builder.add_edge("generate_joke", "explain_joke")
builder.add_edge("explain_joke", END)


memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)

config = {
    "configurable": {
        "thread_id": "1"
    }
}

In [ ]:
result = graph.invoke(
    {
        "topic": "ai"
    },
    config=config
)

print("\nJoke:")
print(result["joke"])

print("\nExplanation:")
print(result["explanation"])


current_state = graph.get_state(config)

print(current_state.values)


history = list(graph.get_state_history(config))

for checkpoint in history:
    print(checkpoint)